Eir 1.0 - ICSD & GSAS-II Compatible

## Overview

This script performs automated Bond Valence Sum (BVS) analysis on multiple crystal structures from:
- **ICSD CIF files** (downloaded from ICSD database)
- **GSAS-II refinement outputs** (.lst files)

The code uses a **3×3×3 supercell approach** to correctly handle high-symmetry space groups (particularly R-3m and P63/mmc common in layered oxides) where atoms on special positions require proper symmetry expansion.

## Key Features

- Automatic coordination number (CN) detection using gap analysis with 0.05Å minimum threshold
- Mixed/partial occupancy support (e.g., Cr₀.₈Ti₀.₂ on same site)
- Bond Valence Sum (BVS) calculation using Gagné & Hawthorne (2015) parameters
- Global Instability Index (GII) calculation for structural stability assessment
- Batch processing of multiple structures with summary CSV output

## Input Files

### Option 1: ICSD CIF Files
- Place CIF files in a folder
- Filenames should contain CollCode (e.g., `YourCustomFileName_CollCode116507.cif`)
- Script will extract CollCode from filename

### Option 2: GSAS-II Files
- Requires both `.lst` file (refinement output) and corresponding `.cif` file
- Place in same directory with matching base names

## How to Use

1. **Set the input directory:**
```python
   input_folder = r"D:\Your\Path\To\CIF\Files"
```

2. **Choose file type:**
```python
   file_type = "icsd"  # or "gsas"
```

3. **Run all cells**

4. **Check outputs:**
   - `batch_bvs_results.csv` - Full results table
   - Console output shows progress and any errors

## Output Columns

- **File**: Input filename
- **CollCode/Phase**: ICSD CollCode or phase name
- **Space_Group**: Crystallographic space group
- **Element**: Chemical element analyzed
- **Site_Label**: Crystallographic site label
- **CN**: Coordination number (from gap analysis)
- **BVS**: Bond Valence Sum
- **Formal_Charge**: Expected oxidation state
- **Deviation**: BVS - Formal_Charge
- **GII**: Global Instability Index (whole structure)

## Interpreting Results

### Coordination Numbers (CN)
- **CN = 6**: Octahedral coordination (expected for most transition metals in layered oxides)
- **CN = 4**: Tetrahedral coordination
- **CN = 1-3**: Likely indicates structural problems or code issues

### Bond Valence Sum (BVS)
- Should approximately equal the formal oxidation state
- **Deviation < ±0.2 v.u.**: Well-bonded
- **Deviation > ±0.3 v.u.**: Over-bonded (negative) or under-bonded (positive)

### Global Instability Index (GII)
- **GII < 0.20 v.u.**: Stable, well-ordered structure
- **0.20 < GII < 0.40 v.u.**: Moderate strain, possibly acceptable
- **GII > 0.40 v.u.**: Significant structural instability
- **GII > 0.60 v.u.**: Severe problems - synthesis likely difficult

**Note:** The traditional 0.2 v.u. threshold is system-dependent (see Miller & Rondinelli, APL Mater. 2023)

## Common Issues

### Problem: All structures show CN = 1-2 instead of 6
- **Solution**: Already fixed in v4 using supercell generation

### Problem: "No bond valence parameters found"
- **Cause**: Element/oxidation state combination not in Gagné & Hawthorne database
- **Solution**: Parameters may need manual addition or use alternative BVS parameters

### Problem: Very high GII (>1.0) for literature structures
- **Possible causes:**
  - Mixed-phase sample (XRD refinement artifact)
  - Metastable structure
  - Genuine severe strain (synthesis should be difficult)

## Technical Details

### Supercell Generation
- Creates 3×3×3 expansion (27 unit cells)
- Applies all symmetry operations to each site
- Uses lattice translations: i,j,k ∈ {-1, 0, 1}
- Position deduplication at 0.05Å grid snapping

### Gap Analysis Algorithm
- Sorts all bond distances for each atom
- Looks for largest gap between consecutive bonds
- **Ignores gaps < 0.05Å** (numerical precision artifacts)
- Defines primary coordination sphere as bonds before largest gap

## Citation

If using this code, please cite:
- **BVS parameters:** Gagné, O.C. & Hawthorne, F.C. (2015) *Acta Cryst. B* **71**, 562-578
- **Structural validation:** Brown, I.D. (2002) *The Chemical Bond in Inorganic Chemistry: The Bond Valence Model*

## Version History

- **v4.0 (Dec 2024)**: Complete rewrite with supercell generation for high-symmetry structures, rebrand to Eir 1.0
- **v3.0**: ICSD CIF compatibility (failed - low CN)
- **v2.0**: Mixed occupancy support
- **v1.0**: Original GSAS-II only version

In [ ]:
# When running for the first time, you'll need to remove the # from the line below in this cell to install the required dependency. Once it has installed, you can add the # symbol back in or delete this cell.

# !pip install gemmi

In [ ]:
#!/usr/bin/env python
# coding: utf-8

"""
Skuld Batch Processor v4.0 - ICSD COMPATIBLE VERSION (GEMMI-BASED)
===================================================================
Automated Bond Valence Sum Analysis with ICSD & GSAS-II Support

Features:
---------
- ICSD CIF support: Calculates bonds using proper crystallographic symmetry (gemmi)
- GSAS-II CIF support: Uses pre-calculated bond data when available  
- Handles partial occupancy (e.g., Na2/3 in NNM)
- Handles mixed occupancy (e.g., Cr0.8Ti0.2 on same site)
- Works with all space groups (R-3m, P63/mmc, C2/m, etc.)
- Tkinter dialog for directory selection (like Skuld 1.5)
- Automatically creates /outputs subfolder in selected directory
- Preserves EXACT Skuld 1.5 CIF parsing and coordination detection logic
- Works in both Jupyter notebooks and command line

Requirements:
-------------
- gemmi (Crystallographic library) for ICSD CIF support
  Install: pip install gemmi
  
  gemmi provides proper crystallographic bond calculation that:
  * Applies space group symmetry operations correctly
  * Handles partial/mixed occupancy
  * Works for all 230 space groups
  * Generates accurate neighbor lists with symmetry codes

Usage:
------
Command line:
    python Skuld_Batch_v4.py

Jupyter notebook:
    from Skuld_Batch_v4 import run_batch_analysis
    run_batch_analysis()

Output:
-------
All results saved to ./outputs/ subfolder within selected directory:
- skuld_batch_results_v4_YYYYMMDD_HHMMSS.csv
- skuld_batch_failed_v4_YYYYMMDD_HHMMSS.csv (if any failures)

Author: Modified from Skuld 1.5 with proper crystallographic ICSD compatibility
Date: December 2024 (v4.0 - gemmi-based proper crystallography)
"""

import os
import sys
import json
import re
import math
from collections import Counter, defaultdict
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# CIF parsing
try:
    from CifFile import ReadCif
except ImportError:
    print("ERROR: PyCifRW not found. Install with: pip install pycifrw")
    sys.exit(1)

# pymatgen for ICSD CIF bond calculation (robust parser + proper crystallography)
try:
    from pymatgen.core import Structure
    from pymatgen.io.cif import CifParser
    PYMATGEN_AVAILABLE = True
    print("✓ pymatgen available - ICSD CIF files supported with proper crystallographic symmetry")
except ImportError:
    PYMATGEN_AVAILABLE = False
    print("⚠ pymatgen not available - ICSD CIFs will fail. Install with: pip install pymatgen")
    print("  pymatgen is REQUIRED for proper handling of:")
    print("   - Partial occupancy (e.g., Na2/3 sites)")
    print("   - Mixed occupancy (e.g., Cr0.8Ti0.2)")
    print("   - All space groups")
    print("   - Robust CIF parsing (handles ICSD formatting quirks)")

# ============================================================================
# DATABASE MANAGER (From Skuld 1.5 - UNCHANGED)
# ============================================================================

class DatabaseManager:
    """Manages external JSON databases for ionic radii and bond valence parameters"""
    
    def __init__(self, data_dir="./Data"):
        self.data_dir = data_dir
        self.shannon_radii = None
        self.bond_valence_params = None
        self.oxidation_states = None
        self.element_categories = None
        
        if not os.path.exists(self.data_dir):
            print(f"Warning: Data directory '{self.data_dir}' not found!")
            print("Please ensure your JSON database files are in the ./Data folder")
    
    def load_shannon_radii(self):
        """Load Shannon-Prewitt ionic radii from JSON file"""
        if self.shannon_radii is not None:
            return self.shannon_radii
            
        filepath = os.path.join(self.data_dir, "shannon_radii.json")
        
        if not os.path.exists(filepath):
            print(f"Shannon radii file not found at {filepath}")
            return None
        
        try:
            with open(filepath, 'r') as f:
                data = json.load(f)
                self.shannon_radii = {}
                for element, valence_dict in data.items():
                    for valence_str, cn_dict in valence_dict.items():
                        valence = int(valence_str)
                        for cn_str, radius in cn_dict.items():
                            cn = int(cn_str)
                            self.shannon_radii[(element, valence, cn)] = radius
                
            print(f"✓ Loaded Shannon radii database with {len(self.shannon_radii)} entries")
        except Exception as e:
            print(f"✗ Error loading Shannon radii: {e}")
            return None
        
        return self.shannon_radii
    
    def load_bond_valence_params(self):
        """Load Gagné & Hawthorne bond valence parameters from JSON file"""
        if self.bond_valence_params is not None:
            return self.bond_valence_params
            
        filepath = os.path.join(self.data_dir, "bvs_parameters.json")
        
        if not os.path.exists(filepath):
            print(f"BVS parameters file not found at {filepath}")
            return None
        
        try:
            import ast
            with open(filepath, 'r') as f:
                data = json.load(f)
                self.bond_valence_params = {}
                for key, params in data.items():
                    # Keys are string representations of tuples: "('Na', 1, 'O')"
                    try:
                        cation, cation_valence, anion = ast.literal_eval(key)
                        self.bond_valence_params[(cation, cation_valence, anion)] = {
                            'R0': params['r0'],  # Note lowercase in JSON
                            'B': params.get('b', 0.37)  # Note lowercase in JSON
                        }
                    except (ValueError, SyntaxError) as e:
                        # Skip malformed keys
                        continue
            
            print(f"✓ Loaded BVS parameters database with {len(self.bond_valence_params)} entries")
        except Exception as e:
            print(f"✗ Error loading BVS parameters: {e}")
            import traceback
            traceback.print_exc()
            return None
        
        return self.bond_valence_params
    
    def load_common_oxidation_states(self):
        """Load common oxidation states from JSON file"""
        if self.oxidation_states is not None:
            return self.oxidation_states
            
        filepath = os.path.join(self.data_dir, "common_oxidation_states.json")
        
        if not os.path.exists(filepath):
            print(f"Oxidation states file not found at {filepath}")
            return None
        
        try:
            with open(filepath, 'r') as f:
                self.oxidation_states = json.load(f)
            
            print(f"✓ Loaded oxidation states database with {len(self.oxidation_states)} elements")
        except Exception as e:
            print(f"✗ Error loading oxidation states: {e}")
            return None
        
        return self.oxidation_states
    
    def load_element_categories(self):
        """Load element categories from JSON file"""
        if self.element_categories is not None:
            return self.element_categories
            
        filepath = os.path.join(self.data_dir, "element_categories.json")
        
        if not os.path.exists(filepath):
            print(f"Element categories file not found at {filepath}")
            return None
        
        try:
            with open(filepath, 'r') as f:
                self.element_categories = json.load(f)
            
            print(f"✓ Loaded element categories database")
        except Exception as e:
            print(f"✗ Error loading element categories: {e}")
            return None
        
        return self.element_categories
    
    def get_ionic_radius(self, element, valence, cn):
        """Get ionic radius for given element, valence, and coordination number"""
        if self.shannon_radii is None:
            self.load_shannon_radii()
        
        if self.shannon_radii is None:
            return None
        
        return self.shannon_radii.get((element, valence, cn))
    
    def get_bv_parameters(self, cation, cation_valence, anion):
        """Get R0 and B parameters for BVS calculation"""
        if self.bond_valence_params is None:
            self.load_bond_valence_params()
        
        if self.bond_valence_params is None:
            return None, None
        
        params = self.bond_valence_params.get((cation, cation_valence, anion))
        if params:
            return params['R0'], params['B']
        return None, None
    
    def get_common_oxidation_states(self, element):
        """Get list of common oxidation states for an element"""
        if self.oxidation_states is None:
            self.load_common_oxidation_states()
        
        if self.oxidation_states is None:
            return []
        
        return self.oxidation_states.get(element, [])
    
    def get_element_category(self, element):
        """Get category for an element (alkali, transition, etc.)"""
        if self.element_categories is None:
            self.load_element_categories()
        
        if self.element_categories is None:
            return "unknown"
        
        for category, elements in self.element_categories.items():
            if element in elements:
                return category
        return "other"

# Initialize global database manager
db_manager = DatabaseManager()
db_manager.load_shannon_radii()
db_manager.load_bond_valence_params()
db_manager.load_common_oxidation_states()
db_manager.load_element_categories()

# ============================================================================
# UTILITY FUNCTIONS (From Skuld 1.5 - UNCHANGED)
# ============================================================================

def parse_value_with_error(value_string):
    """
    Parse CIF value strings that may contain errors in parentheses.
    Example: "1.2345(67)" -> 1.2345, 0.0067
    """
    if not value_string or value_string == '.' or value_string == '?':
        return None, None
    
    match = re.match(r'^([+-]?\d+\.?\d*(?:[eE][+-]?\d+)?)\((\d+)\)$', value_string.strip())
    if match:
        value_str, error_str = match.groups()
        value = float(value_str)
        
        if '.' in value_str:
            decimal_places = len(value_str.split('.')[1].split('e')[0].split('E')[0])
        else:
            decimal_places = 0
        
        error = float(error_str) * (10 ** (-decimal_places))
        return value, error
    else:
        try:
            value = float(value_string.strip())
            return value, None
        except ValueError:
            return None, None

def validate_element_combination(cation, anion):
    """Check if cation-anion combination is chemically reasonable"""
    cation_category = db_manager.get_element_category(cation)
    anion_category = db_manager.get_element_category(anion)
    
    if cation_category == "other" and anion_category == "other":
        return False
    
    return True

# ============================================================================
# BOND DISTANCE CALCULATION FOR ICSD CIF FILES
# ============================================================================

def clean_cif_for_gemmi(cif_file_path):
    """
    Pre-process CIF file to remove lines that cause gemmi parsing errors.
    
    Removes _chemical_name_structure_type which often has UTF-8 encoding issues in ICSD files.
    This tag is not needed for bond calculation.
    
    Returns cleaned CIF as string for gemmi to parse.
    """
    try:
        # Read file with UTF-8 encoding, replacing errors
        with open(cif_file_path, 'r', encoding='utf-8', errors='replace') as f:
            lines = f.readlines()
        
        cleaned_lines = []
        i = 0
        while i < len(lines):
            line = lines[i]
            
            # Check if this line contains the problematic tag
            if '_chemical_name_structure_type' in line:
                # Skip this line
                # Also skip the next line if it doesn't start with _ or # (it's the value)
                i += 1
                if i < len(lines):
                    next_line = lines[i].strip()
                    # If next line is a value (not a tag), skip it too
                    if next_line and not next_line.startswith('_') and not next_line.startswith('#') and not next_line.startswith('loop_') and not next_line.startswith('data_'):
                        i += 1
                continue
            
            cleaned_lines.append(line)
            i += 1
        
        cleaned = ''.join(cleaned_lines)
        # Debug info
        # print(f"  Debug: Removed {len(lines) - len(cleaned_lines)} lines")
        return cleaned
    
    except Exception as e:
        print(f"  Warning: Error cleaning CIF file: {e}")
        return None


def calculate_bonds_from_positions(cif_file_path, max_bond_distance=3.5):
    """
    Calculate bond distances from CIF using proper crystallographic symmetry operations.
    
    Uses gemmi library for robust handling of:
    - Partial occupancy (e.g., Na site 2/3 occupied)  
    - Mixed occupancy (e.g., Cr0.8Ti0.2 on same site)
    - All space groups (R-3m, P63/mmc, C2/m, etc.)
    - Proper symmetry code generation
    
    CRITICAL: ICSD files are small molecule/inorganic structures, not macromolecular.
    Uses SmallStructure, not Structure.
    
    Args:
        cif_file_path: Path to CIF file
        max_bond_distance: Maximum bond distance to consider (Ångströms)
    
    Returns:
        dict with bond data formatted like GSAS-II _geom_bond output, or None if failed
    """
    try:
        import gemmi
    except ImportError:
        print("  ✗ gemmi not available - install with: pip install gemmi")
        print("    gemmi is required for proper crystallographic bond calculation")
        return None
    
    import tempfile
    
    try:
        # Read file with UTF-8 error handling
        cleaned_cif = clean_cif_for_gemmi(cif_file_path)
        if cleaned_cif is None:
            print("  ✗ Could not read CIF file")
            return None
        
        # Write to temporary file
        with tempfile.NamedTemporaryFile(mode='w', suffix='.cif', delete=False, encoding='utf-8') as tmp:
            tmp.write(cleaned_cif)
            tmp_path = tmp.name
        
        try:
            # Read CIF document
            doc = gemmi.cif.read(tmp_path)
            block = doc.sole_block()
            
            # CRITICAL: Use SmallStructure for ICSD (small molecule/inorganic) CIFs
            # NOT Structure (macromolecular)
            small_struct = gemmi.make_small_structure_from_block(block)
            
        finally:
            # Clean up temp file
            import os
            try:
                os.unlink(tmp_path)
            except:
                pass
        
        if not small_struct or len(small_struct.sites) == 0:
            print("  ✗ No atomic sites found in CIF")
            return None
        
        # Setup neighbor search for SmallStructure
        # Use min_dist=0.01 in find_site_neighbors to filter zero-distance bonds
        ns = gemmi.NeighborSearch(small_struct, max_bond_distance).populate()
        
        # Build bond data in GSAS-II format
        bond_data = {
            'bond_label_1': [],
            'bond_label_2': [],
            'bond_distance': [],
            'bond_symmetry_2': []
        }
        
        bond_count = 0
        atom_type_pairs = set()
        
        # Debug: Print all sites found
        print(f"  Debug: Found {len(small_struct.sites)} sites in asymmetric unit:")
        for site in small_struct.sites:
            frac = site.fract
            print(f"    {site.label}: {site.element.name} at ({frac.x:.4f}, {frac.y:.4f}, {frac.z:.4f}) occ={site.occ:.2f}")
        
        # CRITICAL: Generate full supercell for proper coordination detection
        # Gemmi's neighbor search works with asymmetric units, but we need all atoms
        # Generate 3x3x3 supercell to ensure we find all neighbors within cutoff
        print(f"  Generating 3×3×3 supercell for coordination analysis...")
        
        supercell_atoms = []
        
        # For each site in asymmetric unit
        for site in small_struct.sites:
            elem_type = site.element.name
            site_label = site.label
            
            # Apply all symmetry operations
            for op in small_struct.spacegroup.operations():
                # Transform fractional coordinates
                transformed_frac = op.apply_to_xyz([site.fract.x, site.fract.y, site.fract.z])
                
                # Apply lattice translations for 3x3x3 supercell
                for i in range(-1, 2):  # -1, 0, 1
                    for j in range(-1, 2):
                        for k in range(-1, 2):
                            # Add lattice translation
                            final_frac = [
                                transformed_frac[0] + i,
                                transformed_frac[1] + j,
                                transformed_frac[2] + k
                            ]
                            
                            # Convert to Cartesian
                            cart_pos = small_struct.cell.orthogonalize(gemmi.Fractional(*final_frac))
                            
                            supercell_atoms.append({
                                'label': site_label,
                                'element': elem_type,
                                'position': cart_pos,
                                'frac': final_frac,
                                'original_site': site
                            })
        
        print(f"  Generated {len(supercell_atoms)} atoms in supercell")
        
        # Now find neighbors by simple distance calculation
        for site in small_struct.sites:
            site_orth = site.orth(small_struct.cell)
            elem1 = site.label
            elem1_type = site.element.name
            
            neighbors_for_site = []
            
            # Check distance to every atom in supercell
            for atom_dict in supercell_atoms:
                distance = atom_dict['position'].dist(site_orth)
                
                # Skip self and atoms too close/far
                if distance < 0.01 or distance > max_bond_distance:
                    continue
                
                neighbors_for_site.append({
                    'label': atom_dict['label'],
                    'element': atom_dict['element'],
                    'position': atom_dict['position'],
                    'distance': distance
                })
            
            # Deduplicate by position with 0.05Å tolerance  
            # Looser tolerance handles general positions in R-3m better
            unique_neighbors = []
            seen_positions = set()
            
            for neighbor in neighbors_for_site:
                # Round to 0.05Å (50 pm) - catches numerical precision while keeping distinct atoms
                pos_key = (
                    round(neighbor['position'].x / 0.05) * 0.05,
                    round(neighbor['position'].y / 0.05) * 0.05,
                    round(neighbor['position'].z / 0.05) * 0.05
                )
                if pos_key not in seen_positions:
                    seen_positions.add(pos_key)
                    unique_neighbors.append(neighbor)
            
            print(f"  Debug: Site {site.label} ({site.element.name}) has {len(unique_neighbors)} neighbors in supercell")
            
            for neighbor in unique_neighbors:
                elem2 = neighbor['label']
                elem2_type = neighbor['element']
                distance = neighbor['distance']
                atom_type_pairs.add(tuple(sorted([elem1_type, elem2_type])))
                
                # Generate position key with 0.05Å grid snapping (matches deduplication)
                pos_x = round(neighbor['position'].x / 0.05) * 0.05
                pos_y = round(neighbor['position'].y / 0.05) * 0.05
                pos_z = round(neighbor['position'].z / 0.05) * 0.05
                position_key = f"{pos_x:.2f}_{pos_y:.2f}_{pos_z:.2f}"
                symmetry_code = "supercell"
                
                print(f"    -> {elem2} ({elem2_type}) dist={distance:.4f}")
                
                # Store bond
                bond_data['bond_label_1'].append(elem1)
                bond_data['bond_label_2'].append(elem2)
                bond_data['bond_distance'].append(f"{distance:.5f}")
                bond_data['bond_symmetry_2'].append(f"{symmetry_code}|{position_key}")
                bond_count += 1
        
        print(f"  ✓ Calculated {bond_count} bonds using crystallographic symmetry (gemmi)")
        print(f"  ✓ Bond types found: {sorted(atom_type_pairs)}")
        print(f"  ✓ Space group: {small_struct.spacegroup_hm}")
        
        return bond_data
        
    except Exception as e:
        print(f"  ✗ Error calculating bonds with gemmi: {e}")
        import traceback
        traceback.print_exc()
        return None

def is_bond_data_present(data_block):
    """
    Check if a CIF data block contains pre-calculated bond distance data.
    
    Args:
        data_block: PyCifRW data block dictionary
    
    Returns:
        True if bond data present (GSAS-II style), False if missing (ICSD style)
    """
    required_keys = [
        '_geom_bond_atom_site_label_1',
        '_geom_bond_atom_site_label_2',
        '_geom_bond_distance',
        '_geom_bond_site_symmetry_2'
    ]
    
    return all(key in data_block for key in required_keys)

# ============================================================================
# DIRECTORY SELECTION DIALOG
# ============================================================================

def select_directory():
    """
    Opens a tkinter dialog to select a directory containing CIF files.
    Returns the selected directory path or None if cancelled.
    """
    import tkinter as tk
    from tkinter import filedialog
    
    root = tk.Tk()
    root.overrideredirect(True)
    root.attributes('-topmost', True)
    root.withdraw()
    
    print("\n" + "="*80)
    print("SELECT DIRECTORY CONTAINING CIF FILES")
    print("="*80)
    print("A file dialog will open. Please select the directory containing your CIF files.")
    print("(If the dialog appears behind other windows, check your taskbar)")
    
    directory_path = filedialog.askdirectory(
        title="Select Directory Containing CIF Files",
        mustexist=True
    )
    
    root.destroy()
    
    if not directory_path:
        print("\n✗ No directory selected. Exiting.")
        return None
    
    print(f"\n✓ Selected directory: {directory_path}")
    
    # Validate directory contains CIF files
    cif_files = list(Path(directory_path).glob("*.cif"))
    if not cif_files:
        print(f"\n✗ ERROR: No CIF files found in {directory_path}")
        print("Please select a directory containing .cif files")
        return None
    
    print(f"✓ Found {len(cif_files)} CIF files")
    
    return directory_path

# ============================================================================
# CIF PARSING (From Skuld 1.5 - EXACT PRESERVATION)
# ============================================================================

def parse_cif_file(file_path):
    """
    Parse a CIF file and return the CifFile object.
    For ICSD CIFs without bond data, calculates bonds from atomic positions.
    """
    try:
        cf = ReadCif(file_path)
        
        # Check if bonds need to be calculated (ICSD CIFs)
        for block_name, data_block in cf.items():
            if not is_bond_data_present(data_block):
                print(f"  ⚠ No bond data in CIF - calculating from crystallographic symmetry (ICSD mode)...")
                
                # Calculate bonds using ASE
                bond_data = calculate_bonds_from_positions(file_path)
                
                if bond_data is not None:
                    # Add calculated bonds to the data block
                    data_block['_geom_bond_atom_site_label_1'] = bond_data['bond_label_1']
                    data_block['_geom_bond_atom_site_label_2'] = bond_data['bond_label_2']
                    data_block['_geom_bond_distance'] = bond_data['bond_distance']
                    data_block['_geom_bond_site_symmetry_2'] = bond_data['bond_symmetry_2']
                else:
                    print(f"  ✗ Failed to calculate bonds - structure will be skipped")
        
        return cf
    except Exception as e:
        print(f"  ERROR parsing {os.path.basename(file_path)}: {e}")
        return None

def extract_initial_site_data(cif_object):
    """
    EXACT COPY from Skuld 1.5 (lines 808-938)
    Extracts initial atom site data and all bond information,
    including proper handling of symmetry operations for bonds beyond unit cell.
    """
    if not cif_object:
        return {}

    processed_data = {}

    for block_name, data_block in cif_object.items():
        block_data = {
            'unit_cell': {},
            'atom_sites_data': [],
            'all_bond_pairs_found': set(),
            'sites_with_bonds': {}
        }
        processed_data[block_name] = block_data

        # --- General Data Items ---
        block_data['publ_author_name'] = data_block.get("_publ_contact_author_name")
        block_data['gsasii_version'] = data_block.get("_gsas_GSASII_version")
        block_data['pd_block_id'] = data_block.get("_pd_block_id")

        # --- Atom Site Data Processing ---
        label_to_canonical_coords = {} 
        label_to_original_coord_strs = {}
        label_to_type = {}
        
        try:
            labels = data_block['_atom_site_label']
            types = data_block['_atom_site_type_symbol']
            xs_orig = data_block['_atom_site_fract_x'] 
            ys_orig = data_block['_atom_site_fract_y']
            zs_orig = data_block['_atom_site_fract_z']

            if not (labels and types and xs_orig and ys_orig and zs_orig and \
                    len(labels) == len(types) == len(xs_orig) == len(ys_orig) == len(zs_orig)):
                print(f"  No complete _atom_site_loop data found in block {block_name}. Skipping atom site processing for this block.")
                block_data['atom_sites_data'] = [] 
            else:
                for i in range(len(labels)):
                    label = labels[i].strip()
                    atom_type_raw = types[i].strip().rstrip('+-0123456789')
                    atom_type = atom_type_raw.capitalize() if len(atom_type_raw) > 1 else atom_type_raw.upper()
                    
                    try:
                        x = float(xs_orig[i].split('(')[0].strip())
                        y = float(ys_orig[i].split('(')[0].strip())
                        z = float(zs_orig[i].split('(')[0].strip())
                        coords_tuple = (x, y, z)
                        
                        block_data['atom_sites_data'].append({'label': label, 'type': atom_type, 'coords': coords_tuple})
                        label_to_canonical_coords[label] = coords_tuple 
                        label_to_original_coord_strs[label] = (xs_orig[i], ys_orig[i], zs_orig[i])
                        label_to_type[label] = atom_type 
                    except (ValueError, IndexError):
                        print(f"  Warning: Could not parse coordinates for atom {label}. Skipping.")
                        continue
        except KeyError:
            print(f"  No _atom_site_label or related loop data found in block {block_name}. Skipping atom site processing for this block.")
            block_data['atom_sites_data'] = []

        # --- Bond Distances (CRITICAL: includes symmetry code handling) ---
        all_raw_bonds_per_central_site = {} 
        site_metadata = {} 

        bond_label_1s = data_block.get("_geom_bond_atom_site_label_1")
        bond_label_2s = data_block.get("_geom_bond_atom_site_label_2")
        bond_distance_raw_strs = data_block.get("_geom_bond_distance")
        bond_symmetry_2s = data_block.get("_geom_bond_site_symmetry_2")

        if bond_label_1s and bond_label_2s and bond_distance_raw_strs and bond_symmetry_2s and \
           all(len(lst) == len(bond_label_1s) for lst in [bond_label_2s, bond_distance_raw_strs, bond_symmetry_2s]):
            
            for i in range(len(bond_label_1s)):
                atom1_label = bond_label_1s[i].strip()
                atom2_label = bond_label_2s[i].strip()
                dist_str = bond_distance_raw_strs[i].strip()
                atom2_symmetry_code = bond_symmetry_2s[i].strip()  # CRITICAL: preserves symmetry info

                dist_val, dist_err = parse_value_with_error(dist_str)
                
                if dist_val is None:
                    continue 

                atom1_canonical_coords = label_to_canonical_coords.get(atom1_label)
                atom2_canonical_coords = label_to_canonical_coords.get(atom2_label) 

                atom1_type = label_to_type.get(atom1_label)
                atom2_type = label_to_type.get(atom2_label)

                # CRITICAL FIX for ICSD CIFs: If type not found in label_to_type,
                # extract element from label (e.g., "Na1" -> "Na", "Cr1" -> "Cr")
                if not atom1_type and atom1_label:
                    # Extract element symbol from label (strip trailing digits)
                    import re
                    match = re.match(r'^([A-Z][a-z]?)', atom1_label)
                    if match:
                        atom1_type = match.group(1)
                
                if not atom2_type and atom2_label:
                    import re
                    match = re.match(r'^([A-Z][a-z]?)', atom2_label)
                    if match:
                        atom2_type = match.group(1)

                if not atom1_type or not atom2_type:
                    continue 
                
                # Validation
                if not validate_element_combination(atom1_type, atom2_type):
                    print(f"  Warning: Unusual element combination {atom1_type}-{atom2_type}, but including in analysis")
                
                block_data['all_bond_pairs_found'].add(tuple(sorted((atom1_type, atom2_type))))

                central_atom_coords = atom1_canonical_coords
                central_atom_element = atom1_type
                
                if central_atom_coords:
                    if central_atom_coords not in all_raw_bonds_per_central_site:
                        all_raw_bonds_per_central_site[central_atom_coords] = []
                        site_metadata[central_atom_coords] = {
                            'elements': set(),
                            'labels_at_site': set(),
                        }

                    site_metadata[central_atom_coords]['elements'].add(central_atom_element)
                    site_metadata[central_atom_coords]['labels_at_site'].add(atom1_label)
                    
                    # CRITICAL: Store complete bond tuple including symmetry code
                    all_raw_bonds_per_central_site[central_atom_coords].append(
                        (atom1_label, atom2_label, atom1_type, atom2_type, atom2_symmetry_code, dist_val, dist_err)
                    )
        else:
            print(f"  No geometric bond data found or incomplete in block {block_name}.")

        block_data['sites_with_bonds'] = {
            coords: {
                'elements': sorted(list(site_metadata[coords]['elements'])),
                'labels': sorted(list(site_metadata[coords]['labels_at_site'])),
                'raw_bonds_list': raw_bonds_list
            } for coords, raw_bonds_list in all_raw_bonds_per_central_site.items()
        }
    
    return processed_data

# ============================================================================
# AUTOMATED BOND SELECTION (Replaces interactive_bond_type_selection)
# ============================================================================

def auto_select_bond_types(processed_data, target_metals=['Na', 'Cr', 'Ni', 'Mn', 'Fe', 'Ti', 'Sb', 'Mg', 'Al']):
    """
    Automatically select metal-oxygen bond types for analysis.
    Prioritizes M-O bonds where M is a target metal.
    
    Returns: processed_data with 'selected_bond_types_for_analysis' added
    """
    for block_name, block_content in processed_data.items():
        unique_bond_pairs = sorted(list(block_content['all_bond_pairs_found']))
        
        print(f"  DEBUG: Found {len(unique_bond_pairs)} unique bond pairs: {unique_bond_pairs}")
        
        if not unique_bond_pairs:
            block_content['selected_bond_types_for_analysis'] = {}
            continue
        
        # Auto-select M-O bonds
        selected_bond_types_map = {}
        
        for type1, type2 in unique_bond_pairs:
            # Check if this is a metal-oxygen bond
            if 'O' in [type1, type2]:
                metal = type1 if type2 == 'O' else type2
                
                # Check if metal is in target list or is any metal bonded to O
                cat = db_manager.get_element_category(metal)
                is_target = metal in target_metals
                is_metal = cat in ['alkali_metals', 'alkaline_earth', 'transition_metals_3d', 
                                  'transition_metals_4d', 'transition_metals_5d', 'lanthanides', 
                                  'actinides', 'poor_metals']
                
                print(f"  DEBUG: Bond {metal}-O: target={is_target}, metal_category={cat}, is_metal={is_metal}")
                
                if is_target or is_metal:
                    # Metal is central, O is ligand
                    selected_bond_types_map[(metal, 'O')] = {
                        'central_atom_type': metal,
                        'ligand_type': 'O'
                    }
        
        block_content['selected_bond_types_for_analysis'] = selected_bond_types_map
        
        if selected_bond_types_map:
            print(f"  Auto-selected {len(selected_bond_types_map)} M-O bond types: {list(selected_bond_types_map.keys())}")
        else:
            print(f"  No M-O bonds found for auto-selection")
    
    return processed_data

# ============================================================================
# AUTOMATED COORDINATION ANALYSIS (Replaces interactive_review_and_correct)
# ============================================================================

def auto_analyze_coordination(processed_data):
    """
    Automatically analyze coordination numbers and bond lengths.
    Uses the EXACT gap analysis logic from Skuld 1.5 but accepts results automatically.
    
    CRITICAL: Preserves lines 1083-1160 from Skuld 1.5 for proper coordination detection.
    """
    final_data = processed_data

    for block_name, block_content in final_data.items():
        
        if not block_content['sites_with_bonds'] or not block_content['selected_bond_types_for_analysis']:
            continue

        for coords, site_info in block_content['sites_with_bonds'].items():
            central_atom_elements_at_site = site_info['elements']
            
            bonds_for_analysis = {}
            for (central_type_selected, ligand_type_selected) in block_content['selected_bond_types_for_analysis'].keys():
                if central_type_selected in central_atom_elements_at_site:
                    filtered_raw_bonds = [
                        b for b in site_info['raw_bonds_list'] 
                        if b[2] == central_type_selected and b[3] == ligand_type_selected
                    ]
                    if filtered_raw_bonds:
                        bonds_for_analysis[(central_type_selected, ligand_type_selected)] = filtered_raw_bonds

            if not bonds_for_analysis:
                continue

            site_info['cn_and_bonds_by_type'] = {}

            for (central_type, ligand_type), raw_bonds_list_for_pair in bonds_for_analysis.items():
                
                # Report site coordinates for validation
                print(f"\n    Analyzing site at {coords}: {central_type}")
                
                # ======================================================================
                # CRITICAL SECTION: EXACT COPY from Skuld 1.5 (lines 1083-1160)
                # This handles symmetry operations and proper coordination detection
                # ======================================================================
                
                # STEP 1: Identify unique geometric positions using (label, position)
                # For ICSD files, symmetry_code now includes |x_y_z position encoding
                unique_geometric_positions_to_shortest_bond = {}
                
                # DEBUG: Show first few bonds to verify position encoding
                if len(raw_bonds_list_for_pair) <= 10:
                    print(f"  DEBUG: Raw bonds for {central_type}-{ligand_type}:")
                    for bond_tuple in raw_bonds_list_for_pair[:10]:
                        print(f"    {bond_tuple[0]}-{bond_tuple[1]}: {bond_tuple[5]:.4f}Å symm='{bond_tuple[4]}'")
                
                # DEBUG: Track unique positions
                position_debug = set()
                
                for bond_tuple in raw_bonds_list_for_pair:
                    ligand_label = bond_tuple[1]
                    ligand_symmetry_code = bond_tuple[4]  # May include |x_y_z position
                    dist_val = bond_tuple[5]
                    dist_err = bond_tuple[6]

                    # Use Cartesian position if available, else use symmetry code
                    if '|' in ligand_symmetry_code:
                        # ICSD mode: extract position part for uniqueness
                        symm_part, pos_part = ligand_symmetry_code.split('|', 1)
                        geo_position_id = (ligand_label, pos_part)  # Use 3D position
                        position_debug.add(pos_part)
                    else:
                        # GSAS-II mode: use symmetry code
                        geo_position_id = (ligand_label, ligand_symmetry_code)
                    
                    if geo_position_id not in unique_geometric_positions_to_shortest_bond or \
                       dist_val < unique_geometric_positions_to_shortest_bond[geo_position_id][0]:  # Compare with OLD distance [0], not error [1]
                        unique_geometric_positions_to_shortest_bond[geo_position_id] = (dist_val, dist_err)
                
                # DEBUG: Show unique positions found
                if position_debug:
                    print(f"  DEBUG: Found {len(position_debug)} unique positions: {sorted(list(position_debug)[:6])}")

                # STEP 2: Create list of all bonds within coordination sphere
                cn_bonds_list = []
                for (dist_val, dist_err) in unique_geometric_positions_to_shortest_bond.values():
                    cn_bonds_list.append((dist_val, dist_err))
                
                sorted_cn_bonds = sorted(cn_bonds_list, key=lambda x: x[0])

                # STEP 3: Gap analysis to find primary coordination sphere
                # Add minimum gap threshold to avoid splitting coordination due to numerical precision
                MIN_GAP_THRESHOLD = 0.05  # Å - ignore gaps smaller than this
                initial_cn = 0
                primary_sphere_cutoff_dist = 0.0 
                
                if sorted_cn_bonds:
                    max_gap = 0
                    cutoff_index = len(sorted_cn_bonds) 
                    
                    if len(sorted_cn_bonds) > 1:
                        for i in range(len(sorted_cn_bonds) - 1):
                            gap = sorted_cn_bonds[i+1][0] - sorted_cn_bonds[i][0]
                            # Only consider gaps larger than threshold
                            if gap > MIN_GAP_THRESHOLD and gap > max_gap:
                                max_gap = gap
                                cutoff_index = i + 1 
                    
                    initial_cn = cutoff_index
                    if cutoff_index > 0:
                        primary_sphere_cutoff_dist = sorted_cn_bonds[cutoff_index - 1][0]

                # STEP 4: Extract bonds within primary sphere
                initial_bonds_for_display = []
                for bond_tuple in sorted_cn_bonds: 
                    dist_val, dist_err = bond_tuple
                    if dist_val <= primary_sphere_cutoff_dist:
                        initial_bonds_for_display.append({'distance': dist_val, 'error': dist_err})
                
                # STEP 5: Count bond multiplicities
                initial_detailed_bond_counts = []
                if initial_bonds_for_display:
                    grouped_bonds_info = []
                    for bond_dict in initial_bonds_for_display:
                        rounded_dist = round(bond_dict['distance'], 5)
                        original_error = bond_dict['error']
                        grouped_bonds_info.append((rounded_dist, original_error, bond_dict['distance']))

                    bond_counts_map = Counter((r_d, o_e) for r_d, o_e, _ in grouped_bonds_info)
                    sorted_unique_keys = sorted(bond_counts_map.keys())

                    for (dist_val_rounded, dist_err_val) in sorted_unique_keys:
                        count = bond_counts_map[(dist_val_rounded, dist_err_val)]
                        original_dist_val = next((orig_d for r_d, e_v, orig_d in grouped_bonds_info 
                                                  if r_d == dist_val_rounded and e_v == dist_err_val), 
                                                 dist_val_rounded)

                        initial_detailed_bond_counts.append({
                            'count': count, 
                            'distance': original_dist_val, 
                            'error': dist_err_val 
                        })

                # ======================================================================
                # END CRITICAL SECTION
                # ======================================================================
                
                # AUTOMATED DECISION: Accept calculated CN and bonds
                confirmed_cn = initial_cn
                confirmed_bond_counts = initial_detailed_bond_counts
                
                # DIAGNOSTIC OUTPUT: Show bond distances for validation
                if initial_detailed_bond_counts:
                    bond_strs = []
                    for b in initial_detailed_bond_counts:
                        bond_strs.append(f"{b['count']}×{b['distance']:.4f}Å")
                    
                    avg_dist = sum(b['distance']*b['count'] for b in initial_detailed_bond_counts) / sum(b['count'] for b in initial_detailed_bond_counts)
                    
                    # VALIDATION: Check against expected values for O3-type NaCrO2
                    warnings = []
                    if central_type == 'Cr' and ligand_type == 'O':
                        if confirmed_cn != 6:
                            warnings.append(f"⚠️ Expected CN=6, found CN={confirmed_cn}")
                        if avg_dist < 1.95 or avg_dist > 2.05:
                            warnings.append(f"⚠️ Cr-O distance {avg_dist:.3f}Å unusual (expect 1.98-2.01Å)")
                    
                    elif central_type == 'Na' and ligand_type == 'O':
                        if confirmed_cn != 6:
                            warnings.append(f"⚠️ Expected CN=6, found CN={confirmed_cn}")
                        if avg_dist < 2.25 or avg_dist > 2.40:
                            warnings.append(f"⚠️ Na-O distance {avg_dist:.3f}Å unusual (expect 2.30-2.35Å)")
                    
                    # Print diagnostic info
                    print(f"    {central_type}-{ligand_type}: CN={confirmed_cn}, bonds=[{', '.join(bond_strs)}], avg={avg_dist:.4f}Å")
                    for warning in warnings:
                        print(f"      {warning}")
                
                site_info['cn_and_bonds_by_type'][(central_type, ligand_type)] = {
                    'confirmed_cn': confirmed_cn,
                    'confirmed_bond_counts': confirmed_bond_counts
                }
    
    return final_data

# ============================================================================
# AUTOMATED VALENCE ASSIGNMENT (Replaces get_valences_interactively)
# ============================================================================

def auto_assign_valences(processed_data, user_oxidation_states=None):
    """
    Automatically assign oxidation states using user-provided overrides or database defaults.
    
    Args:
        processed_data: Data structure from previous processing steps
        user_oxidation_states: Optional dict of {element: valence} from interactive prompt
    
    Uses user-provided oxidation states if available, otherwise falls back to database.
    """
    for block_name, block_content in processed_data.items():
        if not block_content['sites_with_bonds'] or not block_content.get('selected_bond_types_for_analysis'):
            continue

        for coords, site_info in block_content['sites_with_bonds'].items():
            if 'cn_and_bonds_by_type' not in site_info:
                continue
            
            relevant_elements = set()
            for (central_type, ligand_type) in site_info['cn_and_bonds_by_type'].keys():
                relevant_elements.add(central_type)
                relevant_elements.add(ligand_type)
            
            if not relevant_elements:
                continue

            site_valences = {}
            for element in relevant_elements:
                # PRIORITY 1: User-provided oxidation states
                if user_oxidation_states and element in user_oxidation_states:
                    valence = user_oxidation_states[element]
                
                # PRIORITY 2: Database common states
                else:
                    common_states = db_manager.get_common_oxidation_states(element)
                    
                    if common_states:
                        # Use most common state (first in list)
                        valence = common_states[0]
                    else:
                        # PRIORITY 3: Hardcoded fallback defaults
                        default_valences = {
                            'Na': 1, 'K': 1, 'Li': 1,
                            'O': -2, 'F': -1, 'S': -2,
                            'Ni': 2, 'Mn': 3, 'Fe': 3, 'Cr': 3,
                            'Ti': 4, 'Al': 3, 'Mg': 2, 'Sb': 5
                        }
                        valence = default_valences.get(element, 0)
                
                site_valences[element] = valence
            
            site_info['confirmed_valences'] = site_valences
    
    return processed_data

# ============================================================================
# BVS CALCULATIONS (From Skuld 1.5 - EXACT PRESERVATION)
# ============================================================================

def perform_calculations_and_report(processed_data, b_param_default=0.37):
    """
    Perform BVS, Shannon-Prewitt, and other crystallochemical calculations.
    PRESERVES EXACT calculation logic from Skuld 1.5 (lines 1307-1537).
    """
    calculated_results = {}

    for block_name, block_content in processed_data.items():
        
        if not block_content['sites_with_bonds'] or not block_content.get('selected_bond_types_for_analysis'):
            continue
        
        calculated_results[block_name] = {}

        for coords, site_info in block_content['sites_with_bonds'].items():
            if 'cn_and_bonds_by_type' not in site_info:
                continue

            site_results = {
                'elements': site_info['elements'],
                'labels': site_info['labels'],
                'element_data': {}
            }

            for (central_type, ligand_type), bond_analysis_data in site_info['cn_and_bonds_by_type'].items():
                confirmed_cn = bond_analysis_data['confirmed_cn']
                confirmed_bond_counts = bond_analysis_data['confirmed_bond_counts']
                
                central_valence = site_info['confirmed_valences'].get(central_type)
                ligand_valence = site_info['confirmed_valences'].get(ligand_type)

                if central_valence is None or ligand_valence is None:
                    continue
                
                element_calc_data = {'valence': central_valence, 'ligand_valence': ligand_valence, 'cn': confirmed_cn}

                # --- Shannon-Prewitt Ionic Radii Comparison ---
                if confirmed_bond_counts:
                    central_radius = db_manager.get_ionic_radius(central_type, central_valence, confirmed_cn)
                    ligand_radius = db_manager.get_ionic_radius(ligand_type, ligand_valence, confirmed_cn)

                    if central_radius is not None and ligand_radius is not None:
                        expected_distance = central_radius + ligand_radius
                        
                        element_shannon_diffs = []
                        
                        for bond_info in confirmed_bond_counts:
                            count = bond_info['count']
                            observed_distance = bond_info['distance']
                            bond_error = bond_info.get('error', np.nan)
                            percent_diff = ((observed_distance - expected_distance) / expected_distance) * 100
                            element_shannon_diffs.append({
                                'bond_count': count, 
                                'observed_distance': observed_distance, 
                                'expected_distance': expected_distance, 
                                'percent_diff': percent_diff,
                                'error': bond_error 
                            })
                        
                        element_calc_data['shannon_diffs'] = element_shannon_diffs
                        element_calc_data['cation_radius'] = central_radius
                        element_calc_data['anion_radius'] = ligand_radius

                # --- Ionic Potential ---
                if element_calc_data.get('cation_radius') and element_calc_data['cation_radius'] > 0:
                    ionic_potential = float(abs(central_valence)) / element_calc_data['cation_radius']
                    element_calc_data['ionic_potential'] = ionic_potential

                # --- Bond Valence Sum (BVS) Calculation ---
                if confirmed_bond_counts:
                    r0_val, b_val = db_manager.get_bv_parameters(central_type, central_valence, ligand_type)
                    
                    if r0_val is not None:
                        bvs_total = 0.0
                        
                        for bond_info in confirmed_bond_counts:
                            count = bond_info['count']
                            distance = bond_info['distance']
                            
                            s = math.exp((r0_val - distance) / b_val)
                            bvs_total += (s * count)
                        
                        if central_valence != 0:
                            bvs_percent_diff = ((bvs_total - abs(central_valence)) / abs(central_valence)) * 100
                            
                            element_calc_data['bvs'] = bvs_total
                            element_calc_data['bvs_percent_diff'] = bvs_percent_diff

                # --- Bond Length Distortion (BLD) ---
                if confirmed_bond_counts and confirmed_cn > 0:
                    total_distance_for_avg = sum(b['distance'] * b['count'] for b in confirmed_bond_counts)
                    total_count_for_avg = sum(b['count'] for b in confirmed_bond_counts)
                    
                    if total_count_for_avg > 0:
                        average_observed_distance = total_distance_for_avg / total_count_for_avg
                        
                        sum_of_squared_deviations = 0.0
                        for bond_info in confirmed_bond_counts:
                            distance = bond_info['distance']
                            count = bond_info['count']
                            sum_of_squared_deviations += count * ((distance - average_observed_distance) / average_observed_distance)**2
                        
                        bld = sum_of_squared_deviations / confirmed_cn
                        element_calc_data['bond_length_distortion'] = bld

                site_results['element_data'][central_type] = element_calc_data
            
            calculated_results[block_name][coords] = site_results
            
    return calculated_results

def calculate_global_instability_index(calculated_results):
    """
    Calculate Global Instability Index (GII) - EXACT from Skuld 1.5 (lines 671-776)
    """
    gii_results = {}
    
    for block_name, block_results in calculated_results.items():
        total_bvs_deviation = 0.0
        total_formal_valence = 0.0
        site_contributions = []
        
        for site_coords, site_data in block_results.items():
            if 'element_data' not in site_data:
                continue
                
            for element in site_data['elements']:
                element_data = site_data['element_data'].get(element, {})
                
                if 'bvs' in element_data and element_data['bvs'] is not None:
                    observed_bvs = element_data['bvs']
                    formal_valence = abs(element_data.get('valence', 0))
                    
                    if formal_valence > 0:
                        bvs_deviation = abs(observed_bvs - formal_valence)
                        total_bvs_deviation += bvs_deviation
                        total_formal_valence += formal_valence
                        
                        site_contributions.append({
                            'site': site_coords,
                            'element': element,
                            'formal_valence': formal_valence,
                            'observed_bvs': observed_bvs,
                            'bvs_deviation': bvs_deviation,
                            'bvs_percent_diff': element_data.get('bvs_percent_diff', 0),
                            'element_category': db_manager.get_element_category(element)
                        })
        
        if total_formal_valence > 0:
            gii = total_bvs_deviation / total_formal_valence
            
            if gii < 0.05:
                stability_assessment = "Very Stable"
            elif gii < 0.1:
                stability_assessment = "Stable" 
            elif gii < 0.2:
                stability_assessment = "Moderately Unstable"
            elif gii < 0.3:
                stability_assessment = "Unstable"
            else:
                stability_assessment = "Highly Unstable"
            
            gii_results[block_name] = {
                'gii': gii,
                'stability_assessment': stability_assessment,
                'site_contributions': site_contributions,
                'total_sites_analyzed': len(site_contributions)
            }
        else:
            gii_results[block_name] = {
                'gii': None,
                'stability_assessment': 'Cannot Assess',
                'site_contributions': [],
                'total_sites_analyzed': 0
            }
    
    return gii_results

# ============================================================================
# BATCH PROCESSING ENGINE
# ============================================================================

def process_single_cif(file_path, user_oxidation_states=None):
    """
    Process a single CIF file using the EXACT Skuld 1.5 logic.
    Only the interactive parts are automated.
    
    Args:
        file_path: Path to CIF file
        user_oxidation_states: Optional dict of {element: valence} overrides
                              If provided, these oxidation states will be used instead of database defaults
    
    Returns:
        dict with processing results including success status, metadata, sites data, and GII
    """
    results = {
        'success': False,
        'error': None,
        'metadata': {},
        'sites': [],
        'gii': None,
        'filename': os.path.basename(file_path)
    }
    
    # Parse CIF
    cif_obj = parse_cif_file(file_path)
    if cif_obj is None:
        results['error'] = "Failed to parse CIF file"
        return results
    
    # Process first data block
    block_name = list(cif_obj.keys())[0]
    data_block = cif_obj[block_name]
    
    # Extract metadata
    results['metadata'] = {
        'block_name': block_name,
        'filename': os.path.basename(file_path),
        'chemical_formula': data_block.get("_chemical_formula_sum", "N/A"),
        'space_group': data_block.get("_space_group_name_H-M_alt", 
                                     data_block.get("_symmetry_space_group_name_H-M", "N/A"))
    }
    
    # EXACT Skuld 1.5 workflow
    try:
        # Step 1: Extract site data (EXACT from Skuld 1.5)
        initial_extracted_data = extract_initial_site_data(cif_obj)
        
        if not initial_extracted_data:
            results['error'] = "No data extracted from CIF"
            return results
        
        # Step 2: Auto-select bond types (replaces interactive selection)
        data_with_selected_bonds = auto_select_bond_types(initial_extracted_data)
        
        if not data_with_selected_bonds[block_name].get('selected_bond_types_for_analysis'):
            results['error'] = "No M-O bonds found"
            return results
        
        # Step 3: Auto-analyze coordination (uses EXACT Skuld 1.5 logic)
        final_corrected_data = auto_analyze_coordination(data_with_selected_bonds)
        
        # Step 4: Auto-assign valences (with user overrides if provided)
        final_data_with_valences = auto_assign_valences(final_corrected_data, user_oxidation_states)
        
        # Step 5: Perform calculations (EXACT from Skuld 1.5)
        calculated_results = perform_calculations_and_report(final_data_with_valences)
        
        # Step 6: Calculate GII (EXACT from Skuld 1.5)
        gii_results = calculate_global_instability_index(calculated_results)
        
        # Extract results for export
        if block_name in calculated_results:
            for coords, site_data in calculated_results[block_name].items():
                for element in site_data.get('elements', []):
                    elem_data = site_data['element_data'].get(element, {})
                    
                    # Build site result with proper rounding
                    site_result = {
                        'coords': coords,
                        'element': element,
                        'valence': elem_data.get('valence'),
                        'cn': elem_data.get('cn'),
                        'bvs': round(elem_data.get('bvs'), 4) if elem_data.get('bvs') is not None else None,
                        'bvs_percent_diff': round(elem_data.get('bvs_percent_diff'), 2) if elem_data.get('bvs_percent_diff') is not None else None,
                        'ionic_potential': round(elem_data.get('ionic_potential'), 3) if elem_data.get('ionic_potential') is not None else None,
                        'cation_radius': round(elem_data.get('cation_radius'), 3) if elem_data.get('cation_radius') is not None else None,
                        'bld': round(elem_data.get('bond_length_distortion'), 6) if elem_data.get('bond_length_distortion') is not None else None
                    }
                    
                    # Add Shannon-Prewitt data if available
                    if 'shannon_diffs' in elem_data and elem_data['shannon_diffs']:
                        avg_shannon_diff = np.mean([s['percent_diff'] for s in elem_data['shannon_diffs']])
                        site_result['shannon_diff'] = round(avg_shannon_diff, 2)
                        site_result['avg_bond_length'] = round(np.mean([s['observed_distance'] for s in elem_data['shannon_diffs']]), 4)
                    
                    results['sites'].append(site_result)
        
        # Add GII
        if block_name in gii_results:
            results['gii'] = gii_results[block_name].get('gii')
            results['stability_assessment'] = gii_results[block_name].get('stability_assessment')
        
        results['success'] = True
        
    except Exception as e:
        results['error'] = f"Processing error: {str(e)}"
        import traceback
        print(f"  Detailed error: {traceback.format_exc()}")
    
    return results

# ============================================================================
# INTERACTIVE OXIDATION STATE ASSIGNMENT
# ============================================================================

def collect_unique_elements_from_cifs(directory_path):
    """
    Pre-scan all CIF files to identify unique elements present.
    This allows asking for oxidation states once before batch processing.
    
    Args:
        directory_path: Path to directory containing CIF files
    
    Returns:
        set of element symbols found across all CIFs
    """
    cif_files = list(Path(directory_path).glob("*.cif"))
    unique_elements = set()
    
    print("\n" + "="*80)
    print("PRE-SCANNING CIF FILES TO IDENTIFY ELEMENTS")
    print("="*80)
    
    for cif_file in cif_files:
        try:
            cf = ReadCif(str(cif_file))
            
            for block_name, data_block in cf.items():
                # Get element types from atom site data
                types = data_block.get('_atom_site_type_symbol', [])
                
                for atom_type_raw in types:
                    # Clean up: "Na1" → "Na", "O2-" → "O"
                    atom_type = atom_type_raw.strip().rstrip('+-0123456789')
                    atom_type = atom_type.capitalize() if len(atom_type) > 1 else atom_type.upper()
                    unique_elements.add(atom_type)
        
        except Exception as e:
            continue  # Skip problematic files in pre-scan
    
    print(f"✓ Found {len(unique_elements)} unique elements: {sorted(unique_elements)}")
    return unique_elements


def prompt_for_oxidation_states(elements):
    """
    Interactive prompt for oxidation state assignment.
    Shows smart defaults based on oxide chemistry, allows user override.
    
    Args:
        elements: set of element symbols
    
    Returns:
        dict mapping element → oxidation state {e.g., 'Ti': 4, 'Na': 1}
    """
    # Smart defaults for oxide battery materials
    OXIDE_DEFAULTS = {
        # Alkali metals
        'Li': 1, 'Na': 1, 'K': 1, 'Rb': 1, 'Cs': 1,
        # Alkaline earth
        'Mg': 2, 'Ca': 2, 'Sr': 2, 'Ba': 2,
        # Transition metals (oxide battery cathodes)
        'Ti': 4,   # Ti⁴⁺ most common in oxides
        'V': 5,    # V⁵⁺ in V₂O₅
        'Cr': 3,   # Cr³⁺ in chromites
        'Mn': 4,   # Mn⁴⁺ in MnO₂, battery cathodes (can also be 2, 3)
        'Fe': 3,   # Fe³⁺ slightly more common (can also be 2)
        'Co': 3,   # Co³⁺ in LiCoO₂
        'Ni': 2,   # Ni²⁺ in most cathodes
        'Cu': 2,   # Cu²⁺ most common
        'Zn': 2,   # Zn²⁺ only
        # Other common
        'Al': 3, 'Sb': 5, 'Sn': 4, 'Zr': 4, 'Nb': 5, 'Mo': 6, 'W': 6,
        # Anions
        'O': -2, 'F': -1, 'S': -2, 'Cl': -1, 'Br': -1, 'I': -1,
    }
    
    print("\n" + "="*80)
    print("OXIDATION STATE ASSIGNMENT")
    print("="*80)
    print("Please confirm oxidation states for elements found in your CIF files.")
    print("Defaults are shown in [brackets] - press ENTER to accept, or type new value.")
    print("="*80 + "\n")
    
    oxidation_states = {}
    
    # Sort: anions last, then alphabetical
    anions = sorted([e for e in elements if e in ['O', 'F', 'S', 'Cl', 'Br', 'I']])
    cations = sorted([e for e in elements if e not in anions])
    sorted_elements = cations + anions
    
    for element in sorted_elements:
        # Get default suggestion
        default = OXIDE_DEFAULTS.get(element)
        
        if default is not None:
            prompt_text = f"  {element:>2s} oxidation state [{default:+d}]: "
        else:
            # Check database
            common_states = db_manager.get_common_oxidation_states(element)
            if common_states:
                # For elements with multiple common states, show options
                states_str = ', '.join([f"{s:+d}" for s in common_states])
                default = common_states[0]  # Suggest first
                prompt_text = f"  {element:>2s} oxidation state (common: {states_str}) [{default:+d}]: "
            else:
                prompt_text = f"  {element:>2s} oxidation state [?]: "
                default = None
        
        while True:
            try:
                user_input = input(prompt_text).strip()
                
                if user_input == '' and default is not None:
                    # Accept default
                    oxidation_states[element] = default
                    break
                elif user_input != '':
                    # Parse user input
                    valence = int(user_input)
                    oxidation_states[element] = valence
                    break
                else:
                    # No default and no input
                    print("    ERROR: Please enter an oxidation state")
            
            except ValueError:
                print("    ERROR: Please enter a valid integer (e.g., 4, -2)")
    
    # Show summary
    print("\n" + "="*80)
    print("OXIDATION STATES CONFIRMED:")
    print("="*80)
    for element in sorted_elements:
        valence = oxidation_states[element]
        print(f"  {element:>2s}: {valence:+d}")
    print("="*80 + "\n")
    
    # Final confirmation
    confirm = input("Proceed with these oxidation states? [Y/n]: ").strip().lower()
    if confirm and confirm not in ['y', 'yes', '']:
        print("\nAborted by user")
        return None
    
    return oxidation_states

def batch_process_directory(directory_path):
    """
    Process all CIF files in a directory using EXACT Skuld 1.5 logic.
    Includes interactive oxidation state checkpoint for user control.
    Automatically creates /outputs subfolder for results.
    
    Args:
        directory_path: Path to directory containing CIF files
    
    Returns:
        pandas DataFrame with results, or None if failed
    """
    # Create outputs subdirectory
    output_dir = os.path.join(directory_path, "outputs")
    os.makedirs(output_dir, exist_ok=True)
    print(f"\n✓ Output directory: {output_dir}")
    
    cif_files = list(Path(directory_path).glob("*.cif"))
    
    if not cif_files:
        print(f"No CIF files found in {directory_path}")
        return None
    
    # ============================================================================
    # STEP 1: Pre-scan CIF files to identify all unique elements
    # ============================================================================
    unique_elements = collect_unique_elements_from_cifs(directory_path)
    
    if not unique_elements:
        print("✗ No elements found in CIF files")
        return None
    
    # ============================================================================
    # STEP 2: Interactive oxidation state assignment checkpoint
    # ============================================================================
    user_oxidation_states = prompt_for_oxidation_states(unique_elements)
    
    if user_oxidation_states is None:
        print("\n✗ Batch processing cancelled by user")
        return None
    
    # ============================================================================
    # STEP 3: Process all CIF files with confirmed oxidation states
    # ============================================================================
    print(f"\n{'='*80}")
    print(f"SKULD BATCH PROCESSOR v4.0 - Processing {len(cif_files)} CIF files")
    print(f"ICSD & GSAS-II compatible - EXACT Skuld 1.5 logic")
    print(f"Using user-confirmed oxidation states")
    print(f"{'='*80}\n")
    
    all_results = []
    failed_files = []
    
    for i, cif_file in enumerate(cif_files, 1):
        print(f"[{i}/{len(cif_files)}] Processing {cif_file.name}...", end=" ")
        
        # Process with user-confirmed oxidation states
        result = process_single_cif(str(cif_file), user_oxidation_states)
        
        if result['success']:
            gii_str = f"{result['gii']:.4f}" if result['gii'] is not None else "N/A"
            print(f"✓ ({len(result['sites'])} sites, GII={gii_str})")
            
            # Add each site to results
            for site in result['sites']:
                row = {
                    **result['metadata'], 
                    **site,
                    'gii': round(result['gii'], 4) if result['gii'] is not None else None
                }
                all_results.append(row)
        else:
            print(f"✗ ({result['error']})")
            failed_files.append({
                'filename': cif_file.name,
                'error': result['error']
            })
    
    # ============================================================================
    # STEP 4: Save results to CSV files
    # ============================================================================
    if all_results:
        df = pd.DataFrame(all_results)
        
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        results_file = os.path.join(output_dir, f"skuld_batch_results_v4_{timestamp}.csv")
        df.to_csv(results_file, index=False)
        print(f"\n✓ Results saved to: {results_file}")
        
        # Save oxidation states used for this batch
        oxidation_states_file = os.path.join(output_dir, f"oxidation_states_used_{timestamp}.json")
        with open(oxidation_states_file, 'w') as f:
            import json
            json.dump(user_oxidation_states, f, indent=2, sort_keys=True)
        print(f"✓ Oxidation states saved to: {oxidation_states_file}")
        
        # Save failed files log if any
        if failed_files:
            failed_df = pd.DataFrame(failed_files)
            failed_file = os.path.join(output_dir, f"skuld_batch_failed_v4_{timestamp}.csv")
            failed_df.to_csv(failed_file, index=False)
            print(f"✓ Failed files log saved to: {failed_file}")
            print(f"  ({len(failed_files)} files failed)")
        
        # Print summary statistics
        print(f"\n{'='*80}")
        print("BATCH PROCESSING SUMMARY")
        print(f"{'='*80}")
        print(f"Total CIF files processed: {len(cif_files)}")
        print(f"Successful analyses: {len(all_results)} sites from {df['filename'].nunique()} structures")
        print(f"Failed analyses: {len(failed_files)}")
        
        # Show oxidation states used
        print(f"\nOxidation states used:")
        for element in sorted(user_oxidation_states.keys()):
            valence = user_oxidation_states[element]
            print(f"  {element:>2s}: {valence:+d}")
        
        # Show GII statistics if available
        if 'gii' in df.columns and df['gii'].notna().any():
            print(f"\nGlobal Instability Index (GII) statistics:")
            gii_by_structure = df.groupby('filename')['gii'].first()
            print(f"  Mean GII: {gii_by_structure.mean():.4f}")
            print(f"  Min GII:  {gii_by_structure.min():.4f} ({gii_by_structure.idxmin()})")
            print(f"  Max GII:  {gii_by_structure.max():.4f} ({gii_by_structure.idxmax()})")
        
        print(f"{'='*80}\n")
        
        return df
    else:
        print("\n✗ No successful analyses")
        if failed_files:
            print(f"All {len(failed_files)} files failed. Check error messages above.")
        return None

# ============================================================================
# CONVENIENCE FUNCTIONS
# ============================================================================

def run_batch_analysis():
    """
    Convenience function for easy use in Jupyter notebooks or scripts.
    Opens directory selection dialog and processes all CIF files.
    
    Usage:
        from Skuld_Batch_v4 import run_batch_analysis
        df = run_batch_analysis()
    
    Returns:
        pandas DataFrame with results, or None if failed/cancelled
    """
    print("\n" + "="*80)
    print("SKULD BATCH PROCESSOR v4.0 (ICSD & GSAS-II COMPATIBLE)")
    print("="*80)
    
    # Select directory with tkinter dialog
    directory = select_directory()
    
    if directory is None:
        return None
    
    # Process all CIF files
    results_df = batch_process_directory(directory)
    
    if results_df is not None:
        print("\n" + "="*80)
        print("BATCH PROCESSING COMPLETE")
        print("="*80)
        print(f"Successfully analyzed: {results_df['filename'].nunique()} structures")
        print(f"Total sites analyzed: {len(results_df)}")
        output_dir = os.path.join(directory, "outputs")
        print(f"\nAll results saved to: {output_dir}")
        print("\nNext steps:")
        print("  1. Review the results CSV file")
        print("  2. Check for any failed files in the failed log")
        print("  3. Use Skuld_Batch_Analyzer.py to generate thesis figures")
        
        return results_df
    
    return None

# ============================================================================
# MAIN EXECUTION
# ============================================================================

def main():
    """Main execution function - calls run_batch_analysis"""
    return run_batch_analysis()

if __name__ == "__main__":
    main()